# plumbline — internals

For the maintainer or a reviewer: one section per module, the design choice
**and** the rejected alternative, each claim demonstrated live. Consumers
should read [quickstart.ipynb](quickstart.ipynb) instead.

```
cli.py (argparse)
 |-- score         -> harness.py -> compare.py, executor.py, judge.py, metrics.py
 |-- gate          -> gate.py    -> compare.py, executor.py
 '-- score-routing -> routing.py -> metrics.py

judge.py takes a backend from backends.py: echo (tests) or gemini (extra).
Core is pure stdlib; BigQuery and Gemini live behind optional extras.
```

**Explore it yourself:** open `src/plumbline/` in the file browser. Every
module starts with a docstring stating its purpose and the design decisions —
this notebook proves those claims with running code.

In [ ]:
import subprocess


def sh(command):
    """Run a shell command, print it like a terminal session, return output."""
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output:
        print(output)
    return output


def show(table):
    """Display a pandas table in Jupyter; plain-print under exec()-based tests."""
    try:
        from IPython.display import display

        display(table)
    except ImportError:
        print(table)


# Import the package straight from src/, so the notebook runs from a bare
# checkout with no install step.
import sys

sys.path.insert(0, "../src")

In [ ]:
sh("cd .. && uv run --no-project python examples/build_db.py")

## compare.py — grade rows, not SQL text

`compare()` (`compare.py:102`) is the Spider/BIRD "execution accuracy" idea:
run both queries, compare the rows they return. Rejected alternative: string
or AST comparison of the SQL — it calls equivalent queries different (a JOIN
vs a subquery) and was rejected by the whole benchmark literature.

Two semantics, switched by the gold: without `ORDER BY` a result is a
**multiset** (rows in any order); with `ORDER BY` it is a **sequence**
(positions matter). Rejected alternatives: always-multiset misses a broken
`ORDER BY`; always-positional fails correct unordered queries.

In [ ]:
from plumbline.compare import compare

gold_rows = [{"name": "ann", "age": 30}, {"name": "bob", "age": 40}]
pred_rows = [{"age": 40, "name": "bob"}, {"age": 30, "name": "ann"}]  # rows AND columns reordered

print(compare(gold_rows, pred_rows, "SELECT name, age FROM employee"))                # MATCH: no ORDER BY
print(compare(gold_rows, pred_rows, "SELECT name, age FROM employee ORDER BY age"))   # MISMATCH: order mattered

In [ ]:
# Floats round to 4 decimals: aggregate jitter is not a wrong answer,
# a real numeric difference still is.
print(compare([{"avg": 0.333333333}], [{"avg": 0.3333}], "SELECT AVG(x) FROM t"))   # MATCH
print(compare([{"avg": 0.3334}], [{"avg": 0.3333}], "SELECT AVG(x) FROM t"))        # MISMATCH

In [ ]:
import inspect
from plumbline.compare import row_to_tuple

print(inspect.getsource(row_to_tuple))

Plain-words steps of `row_to_tuple`: normalize every cell to one canonical
string (5, 5.0 and "5" collapse together), **sort** the values — column order
dies right there, at O(c log c) instead of trying O(c!) column arrangements —
then freeze them into a tuple so rows can be hashed and counted.

## executor.py — databases behind one protocol

`Executor` (`executor.py:42`) is a structural protocol: anything with
`.execute(sql) -> ExecResult` qualifies — SQLite in CI, BigQuery in
production, a stub in tests. Rejected alternative: an abstract base class
every backend must inherit from; the protocol needs no import coupling.

Errors become data (`.ok False` plus a message), never exceptions — one bad
query must not kill a 200-question run.

In [ ]:
from plumbline.executor import SQLiteExecutor

executor = SQLiteExecutor("../examples/demo.db")
good = executor.execute("SELECT name, salary FROM employee WHERE dept_id = 1")
print(good.ok, good.rows)
bad = executor.execute("SELECT no_such_column FROM employee")
print(bad.ok, bad.error)

## metrics.py — every rate carries its interval

An accuracy like 19/35 is a point estimate from a noisy sample; the Wilson
score interval (`metrics.py:115`) says how little it pins down. Rejected
alternative: statsmodels gives identical numbers but pulls scipy + numpy +
pandas into a library whose core promise is "stdlib only". The risk of
hand-rolled math is handled by pinning to a published value — recomputed
live here — plus an optional parity test against statsmodels.

In [ ]:
from plumbline.metrics import proportion, wilson_interval

low, high = wilson_interval(19, 35)
print(round(low, 3), round(high, 3))   # published reference: 0.382, 0.695
print(proportion(0, 10))               # lower bound sits at 0, never negative

Comparing two systems: McNemar's exact test (`metrics.py:145`). Only
**discordant** questions — one system right, the other wrong — carry signal;
questions where both agree drop out. `b` and `c` are the two discordant
counts; a small `p_value` means the imbalance is unlikely to be luck.

In [ ]:
from plumbline.metrics import mcnemar_exact, mcnemar_from_vectors

print(mcnemar_exact(b=1, c=8))                                    # 1:8 split -> p ~ 0.04
print(mcnemar_from_vectors([True, True, False], [True, False, False]))  # b=1, c=0 -> no evidence

## judge.py + backends.py — a one-sided tiebreaker

The exact comparator has false negatives: two result sets can mean the same
answer in different shapes. The FLEX-style judge (`judge.py:120`) asks an LLM
about **only** the rows already marked wrong, and can only upgrade them.
Rejected alternative: a two-sided judge that also re-checks MATCHes — it
could silently *lower* scores on LLM whim; instead the one-sidedness is
documented and the number is called upper-leaning.

The backend is an injected `Callable[[str], str]` (`backends.py`). Rejected
alternative: importing an SDK inside the judge — untestable without a
network. `echo_backend` returns a canned reply; `gemini_backend` is the paid
one behind the `[gemini]` extra.

In [ ]:
from plumbline.backends import echo_backend
from plumbline.judge import judge

gold_rows = [{"name": "ann smith"}]
pred_rows = [{"name": "ann"}]   # comparator says MISMATCH; a human might not

backend = echo_backend("VERDICT: EQUIVALENT\nREASON: same person, first-name representation")
print(judge("employee names?", "SELECT full_name FROM employee",
            "SELECT first_name FROM employee", gold_rows, pred_rows, backend))

In [ ]:
def dead_backend(prompt):
    raise RuntimeError("quota exhausted")

print(judge("employee names?", "SELECT full_name FROM employee",
            "SELECT first_name FROM employee", gold_rows, pred_rows, dead_backend))

The second call shows the fail-safe: the backend died, the verdict is empty
and non-equivalent, the row stays wrong, and the run continues. One dead call
must not take down a long run.

## gate.py — the lie it prevents, performed live

The scariest gold defect is the one that makes a **wrong prediction score as
correct**. `SUM` over an empty set returns one row holding NULL — not zero
rows — so a gold whose filter matches nobody produces `[{None}]`, and any
nonsense aggregate that also returns NULL matches it exactly:

In [ ]:
from plumbline.harness import evaluate

broken_gold = "SELECT SUM(salary) FROM employee WHERE hire_year > 2100"        # matches nobody
nonsense_pred = "SELECT SUM(budget) FROM department WHERE name = 'no such team'"

report = evaluate(executor, [("q1", "payroll of future hires?", broken_gold, nonsense_pred)])
print(report.results[0].verdict, report.execution.point)   # MATCH, 1.0 — the lie

In [ ]:
from plumbline.gate import gate

findings = gate([("q1", broken_gold)], executor)
print(findings[0])

`DEGENERATE` is exactly this shape — a single trivial cell (NULL or 0) —
kept separate from `ZERO_ROWS` because the empty-aggregate trap returns one
row, which a zero-row check would miss (`gate.py:74`). The workflow: run the
gate whenever the gold set changes; compute scores only after it is clean.

## harness.py — orchestration and the exclusion rules

`evaluate` (`harness.py:143`) ties executor + comparator + judge + metrics
together. Two outcomes leave the denominator because they say nothing about
the prediction: `GOLD_ERROR` (the answer key itself crashed) and `EMPTY_BOTH`
(both empty — a model that returns nothing would "match"). A crashed
**prediction** is different: that is a real wrong answer and counts. Rejected
alternative: scoring exclusions either way — each direction biases the rate.

In [ ]:
predictions = [
    ("ok",          "names?",   "SELECT name FROM employee",             "SELECT name FROM employee"),
    ("wrong",       "count?",   "SELECT COUNT(*) FROM employee",         "SELECT COUNT(*) FROM department"),
    ("gold_broken", "budgets?", "SELECT no_such_column FROM department", "SELECT budget FROM department"),
]
report = evaluate(executor, predictions)
for r in report.results:
    print(f"{r.id:12} {r.verdict:11} execution_correct={r.execution_correct}")
print(f"scored {report.n_scored} of {report.n_total} — the excluded row never touches the rate")

## routing.py — scoring the table-picking stage

Routing is stage one of the pipeline: pick the single `dataset.table` for the
question. A routing miss guarantees a generation miss, so it is scored
separately (`routing.py`) — the report should say *which* stage failed.

First, normalization: model output arrives decorated, and decoration is not
a wrong answer.

In [ ]:
from plumbline.routing import normalize_table_name

for raw in ["  'sales.orders'  ", "`sales.orders`", "bigquery:sales.orders",
            "analytics-prod.sales.orders"]:
    print(f"{raw!r:38} -> {normalize_table_name(raw)!r}")

Catalog snapping (`routing.py`, `snap_to_catalog`): a model that generates
the table name freely can blend two real names into one that does not exist.
With a catalog, a near-miss snaps to the closest real name — but a genuinely
absent table must stay unmatched. The 0.8 similarity cutoff is that boundary,
shown on both sides:

In [ ]:
from plumbline.routing import SNAP_CUTOFF, snap_to_catalog

catalog = {"sales.orders", "sales.customers", "inventory.stock_levels"}
print(snap_to_catalog("inventory.stock_level", catalog))   # dropped plural -> snaps
print(snap_to_catalog("finance.invoices", catalog))        # genuinely absent -> stays
print(f"cutoff: {SNAP_CUTOFF}")

Snapping is **opt-in** (pass a catalog or nothing) and the report always
carries `n_snapped`. Rejected alternative: always-on snapping — it silently
inflates the rate and makes runs with and without a catalog incomparable.

Two tiers, and the no-exclusions rule (contrast with the harness above): an
empty prediction is a real wrong answer, so the denominator never shrinks.

In [ ]:
from plumbline.routing import evaluate_routing

report = evaluate_routing([
    ("r1", "orders?", "sales.orders", "sales.orders"),
    ("r2", "orders?", "sales.orders", "sales.customers"),
    ("r3", "orders?", "sales.orders", ""),
])
for r in report.results:
    print(f"{r.id}  dataset_correct={r.dataset_correct}  table_correct={r.table_correct}")
print("dataset:", report.dataset)
print("table:  ", report.table)

`r3` (the empty prediction) is wrong on both tiers and still inside `n=3`.
`r2` shows why two tiers exist: right dataset, wrong table — the most common
miss shape, and the fix (better table descriptions) differs from a dataset
miss (better dataset scoping).

## Honest limits, and how to reproduce

- The judge is one-sided, so a judge-adjusted number is upper-leaning; a
  defensible headline still needs a two-sided human audit
  ([METHODOLOGY §4](../docs/METHODOLOGY.md)).
- Single-database execution match has a known false-negative rate (~2.5%
  average in the test-suite paper); true test-suite accuracy is out of scope.
- Routing is top-1 and single-label: no recall@k, no multi-label golds
  ([METHODOLOGY §8](../docs/METHODOLOGY.md)).

Raw inputs live in `examples/`. Reproduce: `make test`, `make example`,
`make example-routing` — all free. The only paid path is
`score --judge-gemini` (one Gemini call per disagreement).